# zip_districts.json builder

Rebuilds `viz/public/zip_districts.json` from the Census Bureau 119th-Congress ZCTA→CD relationship file (`tab20_cd11920_zcta520_natl.txt`, downloaded from `https://www2.census.gov/geo/docs/maps-data/data/rel2020/cd-sld/`).

**Output schema** (consumed by `RepresentativesPage` in `src/App2.jsx`):
- `"01001": "MA-2"` — single-district ZIP, value is a string
- `"11201": ["NY-10", "NY-7", ...]` — multi-district ZIP, value is an array ordered by descending land-area overlap (most-likely district first)

A 1% land-area overlap floor is applied to suppress boundary-line jitter from the Census shapefile.

Census uses CD code `00` for single-district voting states (AK/DE/ND/SD/VT/WY) and `98` for non-voting delegate territories (DC/PR/GU/VI/AS/MP). Both are normalised to integer district `0` to match the `legislators-current.json` convention.

In [1]:
import json
import pathlib
import pandas as pd

HERE = pathlib.Path.cwd()
SRC = HERE / "tab20_cd11920_zcta520_natl.txt"
OUT = HERE.parent / "public" / "zip_districts.json"
OVERLAP_MIN = 0.01

FIPS_TO_USPS = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA",
    "08": "CO", "09": "CT", "10": "DE", "11": "DC", "12": "FL",
    "13": "GA", "15": "HI", "16": "ID", "17": "IL", "18": "IN",
    "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME",
    "24": "MD", "25": "MA", "26": "MI", "27": "MN", "28": "MS",
    "29": "MO", "30": "MT", "31": "NE", "32": "NV", "33": "NH",
    "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI",
    "45": "SC", "46": "SD", "47": "TN", "48": "TX", "49": "UT",
    "50": "VT", "51": "VA", "53": "WA", "54": "WV", "55": "WI",
    "56": "WY", "60": "AS", "66": "GU", "69": "MP", "72": "PR",
    "78": "VI",
}

df = pd.read_csv(
    SRC,
    sep="|",
    dtype={
        "GEOID_CD119_20": str,
        "GEOID_ZCTA5_20": str,
        "AREALAND_PART": "float64",
        "AREALAND_ZCTA5_20": "float64",
    },
    encoding="utf-8-sig",
    usecols=["GEOID_CD119_20", "GEOID_ZCTA5_20", "AREALAND_PART", "AREALAND_ZCTA5_20"],
)

df = df.dropna(subset=["GEOID_CD119_20", "GEOID_ZCTA5_20"])
df = df[df["GEOID_ZCTA5_20"].str.len() == 5]
df = df[df["GEOID_CD119_20"].str.len() == 4]
df = df[df["AREALAND_ZCTA5_20"] > 0]

df["frac"] = df["AREALAND_PART"] / df["AREALAND_ZCTA5_20"]
before = len(df)
df = df[df["frac"] >= OVERLAP_MIN].copy()
print(f"Filtered {before - len(df):,} rows below {OVERLAP_MIN:.0%} land overlap; {len(df):,} kept.")

df["state"] = df["GEOID_CD119_20"].str[:2].map(FIPS_TO_USPS)
df["dist"]  = df["GEOID_CD119_20"].str[2:].replace({"98": "0"}).astype(int)
df = df.dropna(subset=["state"])
df["sd"] = df["state"] + "-" + df["dist"].astype(str)

df = df.sort_values(["GEOID_ZCTA5_20", "frac"], ascending=[True, False])

out = {}
multi_count = 0
for zcta, grp in df.groupby("GEOID_ZCTA5_20", sort=True):
    sds = list(dict.fromkeys(grp["sd"].tolist()))
    out[zcta] = sds[0] if len(sds) == 1 else sds
    if len(sds) > 1:
        multi_count += 1

print(f"Total ZIPs: {len(out):,}")
print(f"Multi-district ZIPs: {multi_count:,} ({multi_count/len(out):.1%})")
for probe in ("01001", "02138", "11201", "30303", "60601", "94103", "99501", "00601"):
    print(f"  probe {probe}: {out.get(probe, '<missing>')}")

with open(OUT, "w") as f:
    json.dump(out, f, separators=(",", ":"))
print(f"Wrote {OUT} ({OUT.stat().st_size:,} bytes)")

Filtered 921 rows below 1% land overlap; 39,226 kept.


Total ZIPs: 33,791
Multi-district ZIPs: 5,121 (15.2%)
  probe 01001: MA-1
  probe 02138: ['MA-5', 'MA-7']
  probe 11201: ['NY-10', 'NY-7']
  probe 30303: GA-5
  probe 60601: IL-7
  probe 94103: CA-11
  probe 99501: AK-0
  probe 00601: PR-0
Wrote /Users/aaditbhatia/Desktop/OBBB Dashboard/viz/public/zip_districts.json (565,542 bytes)
